# TonyPi AprilTag 简单识别测试

本示例参考 `robot_tonypi` 中的实现：使用 `hiwonder.Camera` 获取 TonyPi 的相机画面，使用 `apriltag` 的 `tag36h11` 检测器识别标签。它只做单帧识别，适合作为实验二的起点。

运行前请使机器人保持稳定，并把 AprilTag 放在相机前方。

## 1. 导入库并创建检测器

若提示 `No module named apriltag`，请在当前 Jupyter 内核中运行：`import sys; !{sys.executable} -m pip install apriltag`，安装后重新运行本单元。

In [ ]:
import sys
import time

import cv2
import numpy as np
import apriltag
from IPython.display import Image, display

# 让 Notebook 可以找到 TonyPi 自带的硬件库。
for path in ('/home/pi/TonyPi', '/home/pi/TonyPi/HiwonderSDK'):
    if path not in sys.path:
        sys.path.insert(0, path)

import hiwonder.Camera as Camera

detector = apriltag.Detector(apriltag.DetectorOptions(families='tag36h11'))
print('检测器已创建：tag36h11')

## 2. 打开相机

本单元会启动 TonyPi 相机。只需在开始时运行一次；完成全部测试后，请运行最后的“关闭相机”单元。

In [ ]:
camera = Camera.Camera()
camera.camera_open()
time.sleep(1)
print('相机已打开')

## 3. 定义识别和显示函数

函数会将每个识别到的标签画出绿色边框，并在中心标注其 ID。

In [ ]:
def show_image(frame):
    """在 Notebook 中显示 OpenCV 图像。"""
    ok, encoded = cv2.imencode('.jpg', frame)
    if ok:
        display(Image(data=encoded.tobytes()))


def detect_and_draw(frame):
    """识别一帧图像中的 AprilTag，并返回标注后的图像和结果。"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    tags = detector.detect(gray)
    annotated = frame.copy()

    for tag in tags:
        corners = np.round(tag.corners).astype(np.int32)
        center = tuple(np.round(tag.center).astype(int))
        cv2.polylines(annotated, [corners], True, (0, 255, 0), 2)
        cv2.circle(annotated, center, 4, (0, 0, 255), -1)
        cv2.putText(annotated, f'ID: {tag.tag_id}',
                    (center[0] - 25, center[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    return annotated, tags

## 4. 拍摄一帧并识别

将 `tag36h11` 标签置于相机视野内，运行本单元。输出会显示识别数量、每个标签的 ID 和带标注的图像。若识别数量为 0，请让标签更靠近相机、保持标签完整可见并改善光照，然后再次运行本单元。

In [ ]:
# 先读取几帧，避开刚打开相机时可能出现的旧画面。
for _ in range(5):
    ret, frame = camera.read()
    time.sleep(0.05)

if not ret or frame is None:
    print('未获取到相机画面，请检查相机连接后重新打开相机。')
else:
    annotated, tags = detect_and_draw(frame)
    print(f'识别到 {len(tags)} 个 AprilTag')
    for tag in tags:
        print(f'  ID={tag.tag_id}, 中心坐标={np.round(tag.center, 1)}')
    show_image(annotated)

    filename = f'/home/pi/Pictures/apriltag_{int(time.time())}.jpg'
    cv2.imwrite(filename, annotated)
    print(f'标注结果已保存到：{filename}')

## 5. 关闭相机

完成测试后运行本单元，释放相机设备。

In [ ]:
camera.camera_close()
print('相机已关闭')